# Fall Detection ML Model
## Sensor-Based Fall Detection using Random Forest & 1D CNN

This notebook implements a complete fall detection pipeline using the **Smartphone Human Fall Dataset** from Kaggle:
1. Data Loading & Exploration
2. Exploratory Data Analysis (Mutual Information, Heatmap, Swarm Plots, Skewness)
3. Feature Engineering & Selection
4. Random Forest Classifier (Primary Model)
5. 1D CNN Classifier (Secondary Model)
6. Model Comparison & Saving

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os, glob

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.feature_selection import mutual_info_classif

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Scikit-learn: {__import__("sklearn").__version__}')
print(f'TensorFlow: {tf.__version__}')
print('All libraries loaded successfully!')

## 1. Data Loading & Exploration

The Smartphone Human Fall Dataset contains pre-extracted features from smartphone
accelerometer and gyroscope sensors, covering fall events (SDL, FOL, FKL, BSC)
and activities of daily living (ADLs).

In [ ]:
import kagglehub

sensor_path = kagglehub.dataset_download('saadmansakib/smartphone-human-fall-dataset')
print(f'Dataset path: {sensor_path}')

train_df = pd.read_csv(os.path.join(sensor_path, 'Train.csv'))
test_df = pd.read_csv(os.path.join(sensor_path, 'Test.csv'))

df = pd.concat([train_df, test_df], ignore_index=True)
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

print(f'\nCombined dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nData types:\n{df.dtypes}')
df.head(10)

In [ ]:
print('=== Dataset Summary ===')
print(f'Total samples: {len(df)}')
print(f'Missing values:\n{df.isnull().sum()}\n')
print('Statistical Summary:')
df.describe()

In [ ]:
fall_labels = ['SDL', 'FOL', 'FKL', 'BSC']

print('Activity distribution:')
print(df['label'].value_counts())
print(f'\nBinary target (fall column):')
print(f'  Non-Fall (ADL): {(df["fall"] == 0).sum()}')
print(f'  Fall:           {(df["fall"] == 1).sum()}')
print(f'  Fall ratio:     {df["fall"].mean():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

activity_counts = df['label'].value_counts()
colors = ['#e74c3c' if lbl in fall_labels else '#3498db' for lbl in activity_counts.index]
activity_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Activity Distribution (Red=Falls, Blue=ADLs)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Activity Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

df['fall'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
    colors=['#3498db', '#e74c3c'], labels=['Non-Fall (ADL)', 'Fall'],
    startangle=90, explode=(0, 0.05))
axes[1].set_title('Fall vs Non-Fall Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

feature_cols = ['acc_max', 'gyro_max', 'acc_kurtosis', 'gyro_kurtosis',
                'lin_max', 'acc_skewness', 'gyro_skewness', 'post_gyro_max', 'post_lin_max']
feature_cols = [c for c in feature_cols if c in df.columns]
print(f'\nFeature columns ({len(feature_cols)}): {feature_cols}')

## 2. Exploratory Data Analysis

### 2.1 Mutual Information Scores

Mutual Information measures statistical dependency between each feature and the
fall/non-fall target. High-impact indicators include:
- **acc_max** (peak impact): Falls produce 3.0g-5.0g+ spikes vs 1.5g-2.0g for normal activities
- **post_lin_max** (post-impact state): Monitors linear acceleration after the trigger event

In [ ]:
X_features = df[feature_cols].fillna(df[feature_cols].median())
y_target = df['fall']

mi_scores = mutual_info_classif(X_features, y_target, random_state=42)
mi_df = pd.DataFrame({
    'Feature': feature_cols,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(mi_df['Feature'], mi_df['MI_Score'], color='#2ecc71', edgecolor='black')

for bar, score in zip(bars, mi_df['MI_Score']):
    if score > mi_df['MI_Score'].median():
        bar.set_color('#e74c3c')
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=10)

ax.set_xlabel('Mutual Information Score', fontsize=12)
ax.set_title('Mutual Information Scores - Feature Importance for Fall Detection',
             fontsize=14, fontweight='bold')
ax.axvline(x=mi_df['MI_Score'].median(), color='gray', linestyle='--', alpha=0.7, label='Median')
ax.legend()
plt.tight_layout()
plt.show()

print('\nMutual Information Scores (ranked):')
for _, row in mi_df.sort_values('MI_Score', ascending=False).iterrows():
    print(f'  {row["Feature"]:20s} : {row["MI_Score"]:.4f}')

### 2.2 Correlation Heatmap

Key correlations with the target:
- **post_lin_max (0.86)**: Linear force after impact is an almost perfect indicator of a fall
- **post_gyro_max (0.76)**: Rotational velocity upon fall onset is a strong secondary indicator
- **gyro_max (0.47)**: Overall maximum angular velocity is noisy and non-specific

In [ ]:
corr_df = df[feature_cols + ['fall']].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(
    corr_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    mask=mask, square=True, linewidths=0.5,
    vmin=-1, vmax=1,
    cbar_kws={'label': 'Correlation Coefficient'},
    ax=ax
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with target (fall):')
target_corr = corr_df['fall'].drop('fall').sort_values(ascending=False)
for feat, corr_val in target_corr.items():
    marker = '  << LOW - candidate for removal' if abs(corr_val) < 0.5 else ''
    print(f'  {feat:20s} : {corr_val:+.4f}{marker}')

### 2.3 Swarm Plots

Swarm plots reveal clustering patterns per activity type:
- **Falls (SDL, FOL, FKL, BSC)** cluster in high zones (10-25 units) for post_lin_max and post_gyro_max
- **High-energy ADLs (JOG, JUM)** rarely exceed the 5-unit threshold
- **Sitting activities (CSI, CSO, STN)** occupy the middle ground between 2-8 units

In [ ]:
swarm_features = ['post_lin_max', 'post_gyro_max', 'acc_max']
swarm_features = [f for f in swarm_features if f in df.columns]

sampled_parts = []
for lbl in df['label'].unique():
    subset = df[df['label'] == lbl]
    sampled_parts.append(subset.sample(n=min(len(subset), 60), random_state=42))
plot_df = pd.concat(sampled_parts, ignore_index=True)

label_order = list(df['label'].value_counts().index)
palette = {lbl: '#e74c3c' if lbl in fall_labels else '#3498db' for lbl in label_order}

fig, axes = plt.subplots(1, len(swarm_features), figsize=(7 * len(swarm_features), 8))
if len(swarm_features) == 1:
    axes = [axes]

for ax, feat in zip(axes, swarm_features):
    sns.swarmplot(
        data=plot_df, x='label', y=feat, ax=ax,
        order=label_order, size=3, palette=palette
    )
    ax.set_title(f'{feat} by Activity', fontsize=13, fontweight='bold')
    ax.set_xlabel('Activity Label')
    ax.set_ylabel(feat)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Swarm Plots - Feature Distribution by Activity Type\n(Red=Falls, Blue=ADLs)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2.4 Feature Distribution & Skewness Analysis (Feature Normalization)

Skewness reveals the shape of sensor data distributions:
- **Left-skewed** (acc_max, lin_max): Frequent high-value events critical for fall modeling
- **Right-skewed** (acc_kurtosis, gyro_kurtosis, post_gyro_max): Normal daily activities
- **Symmetric** (post_lin_max): Balanced mix of normal and emergency events

In [ ]:
n_features = len(feature_cols)
n_cols_plot = 3
n_rows_plot = (n_features + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(5 * n_cols_plot, 4 * n_rows_plot))
axes_flat = axes.flatten()

skewness_values = {}

for i, feat in enumerate(feature_cols):
    ax = axes_flat[i]
    skew = df[feat].skew()
    skewness_values[feat] = skew

    sns.histplot(df[feat], kde=True, ax=ax, color='#3498db', edgecolor='black', alpha=0.7)
    ax.axvline(df[feat].mean(), color='red', linestyle='--', linewidth=1.5, label='Mean')
    ax.axvline(df[feat].median(), color='green', linestyle='--', linewidth=1.5, label='Median')
    ax.set_title(f'{feat}\n(Skewness: {skew:.3f})', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Feature Distributions with Skewness Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
skew_df = pd.DataFrame({
    'Feature': list(skewness_values.keys()),
    'Skewness': list(skewness_values.values())
}).sort_values('Skewness')

fig, ax = plt.subplots(figsize=(10, 6))
colors = []
for s in skew_df['Skewness']:
    if s < -0.5:
        colors.append('#e74c3c')
    elif s > 0.5:
        colors.append('#2ecc71')
    else:
        colors.append('#f39c12')

ax.barh(skew_df['Feature'], skew_df['Skewness'], color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1.5)
ax.axvline(x=-0.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)

for i, (feat, skew) in enumerate(zip(skew_df['Feature'], skew_df['Skewness'])):
    ax.text(skew + (0.05 if skew >= 0 else -0.05), i,
            f'{skew:.3f}', va='center', ha='left' if skew >= 0 else 'right', fontsize=10)

ax.set_xlabel('Skewness', fontsize=12)
ax.set_title('Feature Skewness Analysis\n(Red=Left-skewed, Green=Right-skewed, Orange=Symmetric)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSkewness Classification:')
for feat, skew in sorted(skewness_values.items(), key=lambda x: x[1]):
    category = 'LEFT-SKEWED' if skew < -0.5 else ('RIGHT-SKEWED' if skew > 0.5 else 'SYMMETRIC')
    print(f'  {feat:20s} : {skew:+.3f}  [{category}]')

## 3. Feature Engineering & Selection

**Removing `gyro_max`** based on the correlation and MI analysis:
1. Lowest correlation with fall target (~0.47) - non-specific and noisy
2. Rotation ambiguity: high angular velocity arises in non-fall activities
3. `post_gyro_max` already captures angular velocity in context of the impact trigger
4. Reduces dimensionality, mitigates overfitting, improves computational efficiency

In [ ]:
selected_features = [c for c in feature_cols if c != 'gyro_max']
print(f'Removed: gyro_max (low correlation, noisy, rotation ambiguity)')
print(f'Selected features ({len(selected_features)}): {selected_features}')

X = df[selected_features].fillna(df[selected_features].median()).values
y = df['fall'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')
print(f'Train fall ratio: {y_train.mean():.2%}')
print(f'Test fall ratio:  {y_test.mean():.2%}')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('\nFeatures standardized (zero mean, unit variance) for CNN input.')

## 4. Random Forest Classifier (Primary Model)

Random Forest trains an ensemble of decision trees on bootstrap-sampled subsets.
Each tree votes on the classification, and the majority vote determines the final
prediction. Benefits: high accuracy, overfitting resistance, built-in feature importance.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy')
print(f'5-Fold Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Per-fold scores: {[f"{s:.4f}" for s in cv_scores]}')

y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print(f'\n{"="*50}')
print(f'  RANDOM FOREST - TEST RESULTS')
print(f'{"="*50}')
print(f'  Accuracy:  {rf_accuracy:.4f}')
print(f'  Precision: {rf_precision:.4f}')
print(f'  Recall:    {rf_recall:.4f}')
print(f'  F1-Score:  {rf_f1:.4f}')
print(f'{"="*50}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['Non-Fall (ADL)', 'Fall']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Non-Fall', 'Fall'], yticklabels=['Non-Fall', 'Fall'])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title('Random Forest - Confusion Matrix', fontsize=13, fontweight='bold')

importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': selected_features,
    'Importance': importances
}).sort_values('Importance', ascending=True)

axes[1].barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='#e67e22', edgecolor='black')
for i, (imp, feat) in enumerate(zip(feat_imp_df['Importance'], feat_imp_df['Feature'])):
    axes[1].text(imp + 0.005, i, f'{imp:.4f}', va='center', fontsize=10)
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('Random Forest - Feature Importance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print('Feature Importance (ranked):')
for _, row in feat_imp_df.sort_values('Importance', ascending=False).iterrows():
    print(f'  {row["Feature"]:20s} : {row["Importance"]:.4f}')

## 5. 1D CNN Classifier (Secondary Model)

The 1D CNN applies convolutional neural network architecture to sensor features:
- **Conv1D layers**: Extract local patterns using small filters
- **ReLU activation**: Introduces nonlinearity for learning complex relationships
- **MaxPooling1D**: Reduces dimensionality while preserving important signals
- **Dense layers**: Fully connected layers integrate features for classification
- **Sigmoid output**: Binary probability for fall vs non-fall

In [ ]:
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

n_input = X_train_cnn.shape[1]

cnn_model = keras.Sequential([
    layers.Input(shape=(n_input, 1)),

    layers.Conv1D(filters=64, kernel_size=2, activation='relu', padding='same'),
    layers.Conv1D(filters=64, kernel_size=2, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2, padding='same'),
    layers.Dropout(0.3),

    layers.Conv1D(filters=128, kernel_size=2, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2, padding='same'),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTraining completed in {len(history.history["loss"])} epochs')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('1D CNN - Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('1D CNN - Training & Validation Loss', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
y_pred_cnn_prob = cnn_model.predict(X_test_cnn).flatten()
y_pred_cnn = (y_pred_cnn_prob >= 0.5).astype(int)

cnn_accuracy = accuracy_score(y_test, y_pred_cnn)
cnn_precision = precision_score(y_test, y_pred_cnn)
cnn_recall = recall_score(y_test, y_pred_cnn)
cnn_f1 = f1_score(y_test, y_pred_cnn)

print(f'{"="*50}')
print(f'  1D CNN - TEST RESULTS')
print(f'{"="*50}')
print(f'  Accuracy:  {cnn_accuracy:.4f}')
print(f'  Precision: {cnn_precision:.4f}')
print(f'  Recall:    {cnn_recall:.4f}')
print(f'  F1-Score:  {cnn_f1:.4f}')
print(f'{"="*50}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_cnn, target_names=['Non-Fall (ADL)', 'Fall']))

fig, ax = plt.subplots(figsize=(7, 6))
cm_cnn = confusion_matrix(y_test, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Non-Fall', 'Fall'], yticklabels=['Non-Fall', 'Fall'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('1D CNN - Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Comparison & Summary

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Random Forest': [rf_accuracy, rf_precision, rf_recall, rf_f1],
    '1D CNN': [cnn_accuracy, cnn_precision, cnn_recall, cnn_f1]
}).set_index('Metric')

print('=' * 55)
print('        MODEL COMPARISON - FALL DETECTION')
print('=' * 55)
print(comparison.to_string(float_format='{:.4f}'.format))
print('=' * 55)

best_model = 'Random Forest' if rf_f1 >= cnn_f1 else '1D CNN'
print(f'\nBest model (by F1-Score): {best_model}')

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison.index))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['Random Forest'], width, label='Random Forest',
               color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, comparison['1D CNN'], width, label='1D CNN',
               color='#e67e22', edgecolor='black')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison: Random Forest vs 1D CNN', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison.index)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save Trained Models

In [ ]:
models_dir = os.path.join(os.getcwd(), 'models')
os.makedirs(models_dir, exist_ok=True)

rf_path = os.path.join(models_dir, 'random_forest_fall_detector.joblib')
joblib.dump(rf_model, rf_path)
print(f'Random Forest saved:  {rf_path}')

scaler_path = os.path.join(models_dir, 'feature_scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f'Feature scaler saved: {scaler_path}')

cnn_path = os.path.join(models_dir, 'cnn_1d_fall_detector.keras')
cnn_model.save(cnn_path)
print(f'1D CNN saved:         {cnn_path}')

metadata = {
    'selected_features': selected_features,
    'removed_features': ['gyro_max'],
    'fall_labels': fall_labels,
    'rf_metrics': {'accuracy': rf_accuracy, 'precision': rf_precision, 'recall': rf_recall, 'f1': rf_f1},
    'cnn_metrics': {'accuracy': cnn_accuracy, 'precision': cnn_precision, 'recall': cnn_recall, 'f1': cnn_f1},
}
joblib.dump(metadata, os.path.join(models_dir, 'model_metadata.joblib'))
print(f'\nAll models and metadata saved to: {models_dir}')